# EfficientNet-B0 Disease Classifier Training

This Jupyter Notebook trains the **EfficientNet-B0** crop disease classification model on Kaggle GPU.

### Key Specifications:
- **Model Architecture**: EfficientNet-B0 with ImageNet pretrained weights
- **Classification Head**: `Dropout(p=0.3) -> Linear(1280, 3)`
- **Target Classes**: `Healthy` (0), `Powdery` (1), `Rust` (2)
- **Dataset**: Pre-split `Train/` (1,322 images), `Validation/` (60 images), `Test/` (150 images)
- **Accelerations**: Automatic Mixed Precision (AMP) on CUDA GPU
- **Checkpoint Format**: 100% compatible with `backend/ml/disease_classifier/model.py` and `inference.py`

In [ ]:
# Cell 1: Package Imports & CUDA GPU Detection
import os
import sys
import time
import json
import logging
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, UnidentifiedImageError
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('KaggleNotebookTrainer')

# Detect compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print(f'COMPUTE DEVICE: {device}')
if device.type == 'cuda':
    print(f'GPU MODEL     : {torch.cuda.get_device_name(0)}')
    print(f'CUDA VERSION  : {torch.version.cuda}')
else:
    print('WARNING: CUDA not available. Running on CPU.')
print('=' * 60)

In [ ]:
# Cell 2: Hyperparameters & Configuration
CONFIG = {
    'num_classes': 3,
    'class_names': ['Healthy', 'Powdery', 'Rust'],
    'class_to_idx': {'Healthy': 0, 'Powdery': 1, 'Rust': 2},
    'image_size': 224,
    'batch_size': 32,
    'epochs': 20,
    'learning_rate': 0.0001,
    'weight_decay': 0.01,
    'dropout': 0.3,
    'label_smoothing': 0.1,
    'seed': 42,
    'num_workers': 2,
    'pin_memory': True,
    'output_dir': Path('/kaggle/working'),
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
}

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
print(f'Output directory configured at: {CONFIG["output_dir"]}')

In [ ]:
# Cell 3: Automatic Dataset Detection
# Handles the Kaggle 'plant-disease-recognition-dataset' which has
# doubled directories: Train/Train/Healthy/ instead of Train/Healthy/
KAGGLE_INPUT = Path('/kaggle/input')
DATASET_ROOT = None

# Strategy 1: Check known dataset slug
known_path = KAGGLE_INPUT / 'plant-disease-recognition-dataset'
if known_path.exists():
    DATASET_ROOT = known_path

# Strategy 2: Recursive search
if DATASET_ROOT is None and KAGGLE_INPUT.exists():
    for path in KAGGLE_INPUT.rglob('*'):
        if path.is_dir() and (path / 'Train').exists() and (path / 'Validation').exists() and (path / 'Test').exists():
            DATASET_ROOT = path
            break

# Strategy 3: Local development fallback
if DATASET_ROOT is None:
    local_path = Path('../data/raw/plant_disease')
    if (local_path / 'Train').exists():
        DATASET_ROOT = local_path

if DATASET_ROOT is None:
    raise FileNotFoundError('Could not locate dataset. Attach plant-disease-recognition-dataset.')

def resolve_split_dir(root, split_name):
    direct = root / split_name
    if (direct / 'Healthy').is_dir():
        return direct
    doubled = direct / split_name
    if doubled.is_dir() and (doubled / 'Healthy').is_dir():
        return doubled
    for child in direct.iterdir():
        if child.is_dir() and (child / 'Healthy').is_dir():
            return child
    raise FileNotFoundError(f'Cannot find class dirs under {direct}')

SPLIT_DIRS = {s: resolve_split_dir(DATASET_ROOT, s) for s in ['Train', 'Validation', 'Test']}

print(f'Dataset root: {DATASET_ROOT.resolve()}')
for sname, sdir in SPLIT_DIRS.items():
    print(f'  {sname}: {sdir}')

In [ ]:
# Cell 4: Dataset Class & Preprocessing Pipelines
def get_train_transforms(image_size: int):
    return transforms.Compose([
        transforms.Resize((image_size + 32, image_size + 32)),
        transforms.RandomCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=CONFIG['mean'], std=CONFIG['std']),
    ])

def get_eval_transforms(image_size: int):
    return transforms.Compose([
        transforms.Resize((image_size + 32, image_size + 32)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=CONFIG['mean'], std=CONFIG['std']),
    ])

class RobustPlantDiseaseDataset(Dataset):
    def __init__(self, split_dir: Path, transform=None, class_to_idx=None):
        self.split_dir = Path(split_dir)
        self.transform = transform
        self.class_to_idx = class_to_idx or {'Healthy': 0, 'Powdery': 1, 'Rust': 2}
        self.samples = []
        valid_exts = {'.jpg', '.jpeg', '.png', '.webp'}
        
        skipped = 0
        for class_name, class_idx in self.class_to_idx.items():
            class_path = self.split_dir / class_name
            if not class_path.exists():
                logger.warning(f'Class directory missing: {class_path}')
                continue
            for img_file in class_path.iterdir():
                if img_file.suffix.lower() in valid_exts:
                    try:
                        with Image.open(img_file) as img:
                            img.verify()
                        self.samples.append((img_file, class_idx))
                    except Exception:
                        skipped += 1
        logger.info(f'Loaded {len(self.samples)} valid samples from {split_dir.name} ({skipped} corrupt skipped).')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, label

# Construct Dataset & DataLoader instances using resolved split directories
train_dataset = RobustPlantDiseaseDataset(SPLIT_DIRS['Train'], transform=get_train_transforms(CONFIG['image_size']), class_to_idx=CONFIG['class_to_idx'])
val_dataset = RobustPlantDiseaseDataset(SPLIT_DIRS['Validation'], transform=get_eval_transforms(CONFIG['image_size']), class_to_idx=CONFIG['class_to_idx'])
test_dataset = RobustPlantDiseaseDataset(SPLIT_DIRS['Test'], transform=get_eval_transforms(CONFIG['image_size']), class_to_idx=CONFIG['class_to_idx'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

In [ ]:
# Cell 5: Model Factory Architecture
def build_model(num_classes: int = 3, pretrained: bool = True, dropout: float = 0.3):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

model = build_model(num_classes=CONFIG['num_classes'], pretrained=True, dropout=CONFIG['dropout']).to(device)
print('EfficientNet-B0 Model initialized successfully with custom head.')

In [ ]:
# Cell 6: Training Loop with AMP, Scheduler, and Checkpoint Saver
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

best_val_acc = 0.0
best_val_loss = float('inf')
best_checkpoint_payload = None
history = []

print('=' * 75)
print(f'STARTING EFFICIENTNET-B0 TRAINING FOR {CONFIG["epochs"]} EPOCHS')
print('=' * 75)

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()
    
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)
        
    avg_train_loss = train_loss / train_total
    avg_train_acc = train_correct / train_total
    
    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
            
    avg_val_loss = val_loss / val_total
    avg_val_acc = val_correct / val_total
    scheduler.step()
    elapsed = time.time() - t0
    
    print(f'Epoch {epoch:02d}/{CONFIG["epochs"]} | Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc:.4f} | Time: {elapsed:.1f}s')
    
    history.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'train_acc': avg_train_acc,
        'val_loss': avg_val_loss,
        'val_acc': avg_val_acc,
        'learning_rate': scheduler.get_last_lr()[0],
    })
    
    if avg_val_acc > best_val_acc or (avg_val_acc == best_val_acc and avg_val_loss < best_val_loss):
        best_val_acc = avg_val_acc
        best_val_loss = avg_val_loss
        best_checkpoint_payload = {
            'state_dict': model.state_dict(),
            'model_state_dict': model.state_dict(),
            'class_names': CONFIG['class_names'],
            'num_classes': CONFIG['num_classes'],
            'image_size': CONFIG['image_size'],
            'normalization': {'mean': CONFIG['mean'], 'std': CONFIG['std']},
            'model_architecture': 'efficientnet_b0',
            'training_config': CONFIG,
            'epoch': epoch,
            'val_acc': avg_val_acc,
            'val_loss': avg_val_loss,
            'best_val_accuracy': avg_val_acc,
            'seed': CONFIG['seed'],
        }

# Save classifier.pt
pt_path = CONFIG['output_dir'] / 'classifier.pt'
torch.save(best_checkpoint_payload, pt_path)
print(f'\nSaved best model checkpoint to: {pt_path} (Val Acc: {best_val_acc:.4f})')

# Save classifier_classes.json
json_path = CONFIG['output_dir'] / 'classifier_classes.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(CONFIG['class_names'], f, indent=2)
print(f'Saved class names list to: {json_path}')

# Save training_log.csv
df_history = pd.DataFrame(history)
df_history.to_csv(CONFIG['output_dir'] / 'training_log.csv', index=False)
print(f'Saved training log to: {CONFIG["output_dir"] / "training_log.csv"}')

In [ ]:
# Cell 7: Test Set Evaluation using Best Checkpoint
best_model = build_model(num_classes=CONFIG['num_classes'], pretrained=False, dropout=CONFIG['dropout']).to(device)
ckpt = torch.load(pt_path, map_location=device, weights_only=False)
best_model.load_state_dict(ckpt['state_dict'])
best_model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        preds = outputs.argmax(dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)

test_acc = (test_preds == test_labels).mean()
cm = confusion_matrix(test_labels, test_preds)
report_str = classification_report(test_labels, test_preds, target_names=CONFIG['class_names'], digits=4)

print('=' * 75)
print('PLANT-DISEASE DATASET TEST-SET PERFORMANCE EVALUATION')
print('=' * 75)
print(f'Test Top-1 Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\nClassification Report:')
print(report_str)
print('Confusion Matrix:')
print(cm)

# Export test metrics
df_cm = pd.DataFrame(cm, index=CONFIG['class_names'], columns=CONFIG['class_names'])
df_cm.to_csv(CONFIG['output_dir'] / 'confusion_matrix_test.csv')
with open(CONFIG['output_dir'] / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write('Plant-Disease dataset test-set performance\n')
    f.write(f'Top-1 Accuracy: {test_acc:.4f}\n\n')
    f.write(report_str)

In [ ]:
# Cell 8: Plot Training & Validation Curves
plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(df_history['epoch'], df_history['train_loss'], label='Train Loss', marker='o')
plt.plot(df_history['epoch'], df_history['val_loss'], label='Val Loss', marker='s')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(df_history['epoch'], df_history['train_acc'], label='Train Accuracy', marker='o')
plt.plot(df_history['epoch'], df_history['val_acc'], label='Val Accuracy', marker='s')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'training_curves.png', dpi=150)
plt.show()

### ⚠️ Pakistani Field Performance Disclaimer

> **IMPORTANT**: The Plant-Disease dataset (Healthy, Powdery, Rust) is a benchmark demo dataset. High test accuracy on this benchmark **does NOT** directly translate to real-world Pakistani field performance without local data collection, domain adaptation, and human expert validation.

In [ ]:
# Cell 9: Artifact Export Checklist
print('=' * 75)
print('KAGGLE TRAINING COMPLETED — DOWNLOAD THE FOLLOWING FILES:')
print('=' * 75)
files_to_download = [
    'classifier.pt',
    'classifier_classes.json',
    'training_log.csv',
    'classification_report.txt',
    'confusion_matrix_test.csv',
    'training_curves.png',
]
for fname in files_to_download:
    fpath = CONFIG['output_dir'] / fname
    exists = fpath.exists()
    size_kb = fpath.stat().st_size / 1024 if exists else 0
    print(f"  [{'EXISTS' if exists else 'MISSING'}] {fname} ({size_kb:.1f} KB)")

print('\nLocal Deployment Instructions:')
print('  Place "classifier.pt" -> backend/ml/models/classifier.pt')
print('  Place "classifier_classes.json" -> backend/ml/models/classifier_classes.json')